# Generación de texto con GPT-2 en Google Colab

Este modelo GPT-2 se ejecuta **localmente** (en la máquina de Colab, sin API ni internet después de la descarga). Es una versión antigua de GPT creada por OpenAI en 2019, con licencia abierta, y es el punto de partida de las versiones GPT modernas (hoy OpenAI va en la familia **GPT-6**, que veremos más adelante vía API).

A veces existe un conflicto entre `torch`, `torchvision` y `transformers` en algunos entornos de Colab. Dado que GPT-2 es un modelo de texto y **no necesita `torchvision`**, se elimina al inicio.

## Paso 1. Preparar el entorno de Colab

Ejecute esta celda. Si ejecuta el cuaderno y aparece el error, haga:

`Entorno de ejecución > Reiniciar sesión`

Luego ejecute nuevamente desde esta primera celda.

In [1]:
# Celda de corrección para Google Colab
# GPT-2 no necesita torchvision. Se elimina para evitar el conflicto:
# RuntimeError: operator torchvision::nms does not exist

import sys
import subprocess

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision"], check=False)
# Usamos la versión de transformers que trae Colab (sin fijar versión) y añadimos accelerate y safetensors
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers", "accelerate", "safetensors"], check=True)

print("Entorno preparado. Si ya había importado transformers antes, reinicie la sesión y ejecute desde esta celda.")

Entorno preparado. Si ya había importado transformers antes, reinicie la sesión y ejecute desde esta celda.


## Paso 2. Verificar GPU disponible

GPT-2 puede ejecutarse en CPU, pero en GPU es más rápido.

In [2]:
import torch

print("Versión de PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No hay GPU disponible. El modelo se ejecutará en CPU.")

Versión de PyTorch: 2.14.1+cu130
CUDA disponible: True
GPU: Tesla T4


## Paso 3. Cargar tokenizer y modelo GPT-2

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "gpt2"  # Opciones: "gpt2", "gpt2-medium", "gpt2-large"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# GPT-2 no tiene token de padding por defecto. Se usa el token EOS como padding.
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

print(f"Modelo cargado: {MODEL_NAME}")
print(f"Dispositivo usado: {device}")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Modelo cargado: gpt2
Dispositivo usado: cuda


## Paso 4. Generación básica de texto

El modelo recibe un prompt y predice los siguientes tokens.

Se usa el inglés dado que esta versión no es multilinguaje.

In [4]:
prompt = "Once upon a time"

inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.8,
        top_k=50,
        top_p=0.95,
        repetition_penalty=1.15,
        pad_token_id=tokenizer.eos_token_id
    )

texto_generado = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(texto_generado)

Once upon a time, you could become the ruler of this world with two people.

If we were to choose between it or not, how would one want us? We wouldn't be able go back and tell our story like they do because there are so many stories out in these countries that can take place at anytime within four hours; if things went very wrong for an individual who was forced into exile by his


## Paso 5. Función reutilizable para generar texto

Esta función permite cambiar parámetros como temperatura, `top_k`, `top_p` y número máximo de tokens generados.

In [5]:
def generar_texto(
    prompt,
    max_new_tokens=100,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.15
):
    """
    Genera texto usando GPT-2.

    Parámetros:
    - prompt: texto inicial.
    - max_new_tokens: número máximo de tokens nuevos.
    - temperature: controla la aleatoriedad.
    - top_k: limita la selección a los k tokens más probables.
    - top_p: muestreo por núcleo acumulado.
    - repetition_penalty: penaliza repeticiones.
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

## Paso 6. Ejemplos de uso

In [6]:
ejemplo_1 = generar_texto(
    "In an artificial intelligence course, students learn that",
    max_new_tokens=80,
    temperature=0.7
)

print(ejemplo_1)

In an artificial intelligence course, students learn that a neural network (neural net) is the simplest of all systems. It's much simpler than computers in terms for data collection and processing algorithms or any other computing process they have learned to do so but it works with less understanding about how information should be used as well. In fact, these basic concepts are actually not new at AI research labs around the world - rather there was even talk recently by


In [7]:
ejemplo_2 = generar_texto(
    "Natural language processing is important because",
    max_new_tokens=80,
    temperature=0.9
)

print(ejemplo_2)

Natural language processing is important because it allows you to interpret expressions without relying on their grammar. For instance, in this example:
\begin{align*} {2}" / \textstyle "<a href="1">"/3</A>"; 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33


## Paso 7. Comparación rápida de parámetros

- Temperatura baja: texto más conservador.
- Temperatura alta: texto más creativo, pero puede ser menos coherente.

In [8]:
prompt = "The main difference between a lion and a cat is"

print("=== Temperatura 0.3 ===")
print(generar_texto(prompt, temperature=0.3, max_new_tokens=70))

print("=== Temperatura 1.0 ===")
print(generar_texto(prompt, temperature=1.0, max_new_tokens=70))

=== Temperatura 0.3 ===
The main difference between a lion and a cat is that lions are more aggressive than cats.
A Lion: A tiger has the ability to attack humans, but it's not as powerful or strong in its bite-and--as with tigers --the claws of an elephant can be used for biting human beings (see "How To Bite An Elephant"). It also doesn't have any special abilities like elephants
=== Temperatura 1.0 ===
The main difference between a lion and a cat is the size of their paws. The big cats don't have very large paw pads but rather, they get them all covered in fur during training for long periods as it allows me to catch up on how much I've learned thus far without getting snobby at my feet (which are still quite active). Cats will do this because what would happen if you


## Nota

GPT-2 es útil para enseñar la arquitectura GPT y la generación autoregresiva de texto. Sin embargo, sus respuestas son mucho menos precisas que las de modelos recientes: tiene solo 124 millones de parámetros (la versión base) y fue entrenado en 2019, solo con texto en inglés.

Para comparar, en los siguientes cuadernos usaremos modelos actuales a través de API: **gpt-oss** (modelo abierto de OpenAI, vía Groq), **GPT-6** (OpenAI) y **Gemini 3.8** (Google).